# ICT Stratejisi — Geçmiş Veri Backtest

Bu notebook, canlı sistemle **aynı sinyal motorunu** (`backtest/ict` + `engine.py`) ve
**aynı veri kaynağını** (Binance **Futures** klines — `fapi.binance.com`) kullanır.

Yaptığı işler:
1. İki tarih arasında stratejinin **açtığı işlemleri** (sinyalleri) listeler.
2. Her işlem için **backtest** yürütür: bar-bar ilerleyip **TP mi SL mi** olduğunu bulur.
3. Özet metrikleri (win-rate, R, sembol/strateji kırılımı) gösterir.

> Not: Sinyaller `[BASLANGIC, BITIS]` aralığında üretilir; ama TP/SL'in gerçekleşebilmesi
> için simülasyon bitiş tarihinden sonraki veriyle de ilerletilir (lookforward tamponu).

## 0. Kurulum
Gerekli paketler (sistemde yoksa):

In [ ]:
# Gerekirse bağımlılıkları kur (ilk çalıştırmada bir kez):
# %pip install pandas numpy requests python-dateutil pytz click
import sys, os
from pathlib import Path

# Bu notebook backtest/ klasöründe olduğu için ict paketi ve engine import edilebilsin:
NB_DIR = Path.cwd()
if (NB_DIR / 'engine.py').exists():
    BACKTEST_DIR = NB_DIR
elif (NB_DIR / 'backtest' / 'engine.py').exists():
    BACKTEST_DIR = NB_DIR / 'backtest'
else:
    raise RuntimeError('engine.py bulunamadi. Notebook backtest/ klasorunden calistirilmali.')
sys.path.insert(0, str(BACKTEST_DIR))
print('backtest dir:', BACKTEST_DIR)

: 

## 1. Girdiler (Parametreler)
Tarih aralığını ve sembolleri buradan ayarla.

In [1]:
# === GIRDI: iki tarih arasi ===
START_DATE = '2026-05-01'   # dahil (UTC)
END_DATE   = '2026-05-22'   # dahil (UTC) - bu tarihe kadar ACILAN islemler

# Primary timeframe ve HTF — canli sistemle ayni (live_runner.py)
INTERVAL = '1h'

# Hangi semboller? Bos liste => canli uretim seti (live_runner.SYMBOLS)
SYMBOLS = []   # orn: ['BTCUSDT', 'ETHUSDT']  |  [] => tum uretim sembolleri

# Strateji seti:
#   'live' => B/C/D/E (canli uretim, strategy_a haric)
#   'all'  => A/B/C/D/E (klasik backtest)
STRATEGY_SET = 'live'

# Motor isinma (warmup) suresi: sinyallerin dogru uretilmesi icin
# START_DATE oncesinden cekilecek gecmis veri. live_runner 6000h kullaniyor.
WARMUP_HOURS = 6000

# Lookforward tamponu: END_DATE'e yakin acilan islemlerin TP/SL'e ulasabilmesi icin
# bitisten sonra eklenecek veri (gun).
LOOKFORWARD_DAYS = 14

# Veri kaynagi: True => Binance FUTURES (uretimle ayni). Bos donerse spot'a duser.
FUTURES = True

## 2. Motoru ve veri katmanini yukle

In [2]:
import pandas as pd

from ict.data import fetch_klines
from engine import run_engine
from backtester import simulate_signals, compute_metrics, TradeStatus
from ict.strategies import (
    STRATEGIES, strategy_a, strategy_b, strategy_c, strategy_d, strategy_e,
)

# Uretim sembol seti (live_runner.py ile ayni liste)
PROD_SYMBOLS = [
    'BTCUSDT', 'ETHUSDT', 'SOLUSDT', 'BNBUSDT', 'DOGEUSDT']

use_symbols = SYMBOLS if SYMBOLS else PROD_SYMBOLS
strats = [strategy_b, strategy_c, strategy_d, strategy_e] if STRATEGY_SET == 'live' else STRATEGIES

start_ts = pd.Timestamp(START_DATE, tz='UTC')
end_ts   = pd.Timestamp(END_DATE, tz='UTC') + pd.Timedelta(days=1)  # END gunu dahil
fetch_start = start_ts - pd.Timedelta(hours=WARMUP_HOURS)
fetch_end   = end_ts + pd.Timedelta(days=LOOKFORWARD_DAYS)

print(f'Sembol sayisi : {len(use_symbols)}')
print(f'Strateji seti : {STRATEGY_SET} ({len(strats)} strateji)')
print(f'Sinyal penceresi : {start_ts}  ->  {end_ts}')
print(f'Veri cekme araligi: {fetch_start}  ->  {fetch_end}  (futures={FUTURES})')

Sembol sayisi : 5
Strateji seti : live (4 strateji)
Sinyal penceresi : 2026-05-01 00:00:00+00:00  ->  2026-05-23 00:00:00+00:00
Veri cekme araligi: 2025-08-24 00:00:00+00:00  ->  2026-06-06 00:00:00+00:00  (futures=True)


## 3. Veriyi cek, sinyalleri uret, backtest et
Her sembol icin: futures veri -> motor -> aralikta acilan sinyaller -> TP/SL simulasyonu.

In [3]:
def run_symbol(symbol):
    """Tek sembol icin acilan islemleri ve TP/SL sonuclarini dondurur."""
    df = fetch_klines(
        symbol, INTERVAL,
        int(fetch_start.timestamp() * 1000),
        int(fetch_end.timestamp() * 1000),
        futures=FUTURES,
    )
    if df.empty:
        print(f'  {symbol}: veri yok')
        return []

    # Bar suresi (1h => 1 saat). df.index = bar OPEN zamanlari.
    bar_dur = (df.index[1] - df.index[0]) if len(df) > 1 else pd.Timedelta(0)
    n = len(df)

    def bar_close(idx):
        return (df.index[idx] + bar_dur) if (idx is not None and 0 <= idx < n) else None

    # Tum veri uzerinde motoru calistir (warmup dogru calissin)
    signals = run_engine(symbol, df, strategies=strats)

    # Sadece [START, END] araliginda ACILAN sinyaller
    window = [s for s in signals if start_ts <= s.timestamp < end_ts]
    if not window:
        print(f'  {symbol}: aralikta sinyal yok ({len(df)} bar)')
        return []

    # Backtest: bu sinyalleri TUM df uzerinde ilerlet (lookforward dahil) -> TP/SL
    results = simulate_signals(window, df)

    rows = []
    for r in results:
        s = r.signal
        # Sonucun (TP/SL/expired/invalidated) gerceklestigi barin indeksi:
        if r.status.value == 'expired':
            outcome_idx = s.expires_at_index   # entry hic dolmadi -> sure bitis bari
        else:
            outcome_idx = r.exit_index         # tp/sl/invalidated; active'de None
        rows.append({
            'symbol': s.symbol,
            'strategy': s.strategy,
            'signal_time': s.timestamp,                 # sinyal barinin OPEN zamani (DB'deki deger)
            'signal_emit_time': s.timestamp + bar_dur,  # sinyalin GERCEKTEN olustugu an = bar CLOSE
            'direction': s.direction.upper(),
            'entry': s.entry,
            'sl': s.stop_loss,
            'tp': s.take_profit_1,
            'rr': round(s.risk_reward_1, 2) if s.risk_reward_1 else None,
            'quality': s.quality,
            'score': s.confluence_score,
            'outcome': r.status.value,                  # tp_hit / sl_hit / expired / invalidated / active
            'entry_time': bar_close(r.entry_index),     # entry'nin doldugu barin kapanisi
            'outcome_time': bar_close(outcome_idx),     # TP/SL/expiry barinin kapanisi (touch bu bar icinde)
            'pnl_r': round(r.pnl_r, 3) if r.pnl_r is not None else None,
            'pnl_pct': round(r.pnl_pct * 100, 3) if r.pnl_pct is not None else None,
            'bars_to_exit': r.bars_to_exit,
        })
    print(f'  {symbol}: {len(window)} acilan islem')
    return rows


all_rows = []
for sym in use_symbols:
    print(f'=== {sym} ===')
    try:
        all_rows.extend(run_symbol(sym))
    except Exception as e:
        print(f'  HATA: {e}')

trades = pd.DataFrame(all_rows)
if not trades.empty:
    trades = trades.sort_values('signal_time').reset_index(drop=True)
print(f'\nTOPLAM acilan islem: {len(trades)}')

=== BTCUSDT ===
  BTCUSDT: 30 acilan islem
=== ETHUSDT ===
  ETHUSDT: 23 acilan islem
=== SOLUSDT ===
  SOLUSDT: 23 acilan islem
=== BNBUSDT ===
  BNBUSDT: 31 acilan islem
=== DOGEUSDT ===
  DOGEUSDT: 26 acilan islem

TOPLAM acilan islem: 133


## ℹ️ Sinyal zamanı: OPEN mı CLOSE mu?

`signal_time` (ör. `2026-05-01 18:00:00`) o barın **AÇILIŞ (open)** zamanıdır — veritabanına da bu değer yazılır
([ict/data.py](ict/data.py) Binance kline'ın open-time alanını index yapar).

Ama motor look-ahead yapmaz: kararı barın **kapanış (close)** verisiyle verir ve simülasyon `bar_index + 1`'den
ilerler ([backtester.py](backtester.py)). Yani sinyal **bar kapandığında** geçerli/işlenebilir olur.

→ `signal_time = 18:00` olan 1h sinyali, bar **19:00'da kapanınca** oluşur. Bu anı tabloda `signal_emit_time` kolonunda
ayrıca gösteriyoruz (= 19:00). Bu, repodaki *"UI labels anchor on bar CLOSE"* davranışıyla aynıdır.

`entry_time` / `outcome_time` kolonları ise işlemin girdiği ve TP/SL/expiry olduğu **barın kapanış** zamanıdır
(fiyat dokunuşu o bar içinde bir yerde gerçekleşir).

## 4. Acilan Islemler (iki tarih arasi)
Stratejinin `START_DATE` ile `END_DATE` arasinda actigi tum islemler ve sonuclari.

In [4]:
pd.set_option('display.max_rows', 500)
pd.set_option('display.width', 200)
trades

,symbol,strategy,signal_time,signal_emit_time,direction,entry,sl,tp,rr,quality,score,outcome,entry_time,outcome_time,pnl_r,pnl_pct,bars_to_exit
0,ETHUSDT,D_silver_bullet,2026-05-01 14:00:00+00:00,2026-05-01 15:00:00+00:00,SHORT,2322.545000,2330.525571,2275.460000,5.90,medium,8,expired,NaT,2026-05-01 18:00:00+00:00,NaN,NaN,NaN
1,DOGEUSDT,D_silver_bullet,2026-05-01 14:00:00+00:00,2026-05-01 15:00:00+00:00,LONG,0.108455,0.108117,0.112000,10.47,medium,8,sl_hit,2026-05-01 17:00:00+00:00,2026-05-01 17:00:00+00:00,-1.000,-0.312,2.0
2,SOLUSDT,D_silver_bullet,2026-05-01 14:00:00+00:00,2026-05-01 15:00:00+00:00,SHORT,84.975000,85.181286,82.949286,9.82,medium,8,expired,NaT,2026-05-01 18:00:00+00:00,NaN,NaN,NaN
3,BNBUSDT,D_silver_bullet,2026-05-01 14:00:00+00:00,2026-05-01 15:00:00+00:00,SHORT,625.305000,626.651429,616.300000,6.69,medium,8,expired,NaT,2026-05-01 18:00:00+00:00,NaN,NaN,NaN
4,BTCUSDT,D_silver_bullet,2026-05-01 18:00:00+00:00,2026-05-01 19:00:00+00:00,LONG,77717.750000,77496.221429,79455.000000,7.84,medium,8,tp_hit,2026-05-01 21:00:00+00:00,2026-05-04 02:00:00+00:00,7.842,2.235,55.0
5,ETHUSDT,D_silver_bullet,2026-05-01 18:00:00+00:00,2026-05-01 19:00:00+00:00,SHORT,2322.545000,2330.581714,2275.460000,5.86,medium,8,expired,NaT,2026-05-01 22:00:00+00:00,NaN,NaN,NaN
6,DOGEUSDT,D_silver_bullet,2026-05-01 18:00:00+00:00,2026-05-01 19:00:00+00:00,LONG,0.107040,0.106426,0.110923,6.32,medium,8,expired,NaT,2026-05-01 22:00:00+00:00,NaN,NaN,NaN
7,BNBUSDT,D_silver_bullet,2026-05-01 18:00:00+00:00,2026-05-01 19:00:00+00:00,SHORT,625.305000,626.637857,616.347857,6.72,medium,8,expired,NaT,2026-05-01 22:00:00+00:00,NaN,NaN,NaN
8,SOLUSDT,D_silver_bullet,2026-05-01 18:00:00+00:00,2026-05-01 19:00:00+00:00,SHORT,84.975000,85.179714,82.980714,9.74,medium,8,expired,NaT,2026-05-01 22:00:00+00:00,NaN,NaN,NaN
9,BTCUSDT,D_silver_bullet,2026-05-02 14:00:00+00:00,2026-05-02 15:00:00+00:00,LONG,78251.900000,78189.364286,78879.900000,10.04,medium,8,expired,NaT,2026-05-02 18:00:00+00:00,NaN,NaN,NaN


## 5. TP mi SL mi? — Sonuc Dagilimi

In [ ]:
if trades.empty:
    print('Aralikta hic islem acilmamis.')
else:
    counts = trades['outcome'].value_counts()
    print('Sonuc dagilimi:')
    print(counts.to_string())

    tp = int((trades['outcome'] == 'tp_hit').sum())
    sl = int((trades['outcome'] == 'sl_hit').sum())
    closed = tp + sl
    print('\n--- Kapanan islemler ---')
    print(f'TP (kazanan): {tp}')
    print(f'SL (kaybeden): {sl}')
    if closed:
        print(f'Win-rate    : {tp / closed * 100:.1f}%')
    print(f'Toplam R    : {trades["pnl_r"].fillna(0).sum():.2f}R')

### 5b. Sadece TP olanlar / sadece SL olanlar

In [ ]:
if not trades.empty:
    display(trades[trades['outcome'] == 'tp_hit'])
    display(trades[trades['outcome'] == 'sl_hit'])

## 6. Ozet Metrikler (engine ile ayni hesap)

In [ ]:
if not trades.empty:
    # compute_metrics ham TradeResult ister; tum sembolleri tekrar toplayalim
    grp = trades.groupby('outcome').size().to_dict()
    summary = {
        'toplam_acilan': len(trades),
        'tp_hit': grp.get('tp_hit', 0),
        'sl_hit': grp.get('sl_hit', 0),
        'expired': grp.get('expired', 0),
        'invalidated': grp.get('invalidated', 0),
        'active (hala acik)': grp.get('active', 0),
        'toplam_R': round(trades['pnl_r'].fillna(0).sum(), 3),
    }
    closed = summary['tp_hit'] + summary['sl_hit']
    summary['win_rate'] = round(summary['tp_hit'] / closed, 4) if closed else 0.0
    summary['expectancy_R'] = round(summary['toplam_R'] / closed, 4) if closed else 0.0
    import pprint; pprint.pprint(summary)

### 6b. Sembol ve strateji kirilimi

In [ ]:
if not trades.empty:
    closed_df = trades[trades['outcome'].isin(['tp_hit', 'sl_hit'])].copy()
    closed_df['win'] = (closed_df['outcome'] == 'tp_hit').astype(int)

    def breakdown(col):
        g = closed_df.groupby(col).agg(
            n=('outcome', 'size'),
            tp=('win', 'sum'),
            toplam_R=('pnl_r', 'sum'),
        )
        g['sl'] = g['n'] - g['tp']
        g['win_rate'] = (g['tp'] / g['n']).round(3)
        g['toplam_R'] = g['toplam_R'].round(2)
        return g[['n', 'tp', 'sl', 'win_rate', 'toplam_R']].sort_values('toplam_R', ascending=False)

    print('--- Sembol bazinda ---')
    display(breakdown('symbol'))
    print('--- Strateji bazinda ---')
    display(breakdown('strategy'))

### 6c. Kumulatif R egrisi (opsiyonel)

In [ ]:
if not trades.empty:
    closed_df = trades[trades['outcome'].isin(['tp_hit', 'sl_hit'])].copy()
    closed_df = closed_df.sort_values('signal_time')
    closed_df['cum_R'] = closed_df['pnl_r'].fillna(0).cumsum()
    try:
        ax = closed_df.set_index('signal_time')['cum_R'].plot(
            figsize=(11, 4), title='Kumulatif R (kapanan islemler)', grid=True)
        ax.set_ylabel('R')
    except Exception as e:
        print('Grafik atlandi (matplotlib yok?):', e)
        print(closed_df[['signal_time', 'symbol', 'outcome', 'pnl_r', 'cum_R']].to_string())

---
**Sonuc kolonu sozlugu:**
- `tp_hit` — Take-Profit'e ulasti (kazanc)
- `sl_hit` — Stop-Loss'a ulasti (zarar, -1R)
- `expired` — Limit emir suresi (EXPIRY_MINUTES) icinde entry dolmadi
- `invalidated` — Entry dolmadan TP/SL bolgesine girildi (sinyal gecersiz)
- `active` — Entry doldu ama veri bitene kadar TP/SL olmadi (hala acik)

**Zaman kolonlari:**
- `signal_time` — sinyal barinin **OPEN** zamani (veritabanindaki deger)
- `signal_emit_time` — sinyalin gercekten olustugu an = sinyal barinin **CLOSE** zamani
- `entry_time` — entry'nin doldugu barin kapanis zamani (entry dolmadiysa bos)
- `outcome_time` — **TP / SL / expiry'nin gerceklestigi** barin kapanis zamani (`active`'de bos)

`pnl_r`: R cinsinden sonuc (1R = baslangic riski = |entry - sl|). `pnl_pct`: entry'ye gore yuzde hareket.
`bars_to_exit`: sinyal barindan cikis barina kadar gecen bar sayisi.